# Adaptive HRC Experiment Suite

Notebook interface for the clean `src.experiments` runner. The suite is organized by logical experiment names rather than numbered tests, with separate main and appendix registries. Each run writes one `result.json` per seed folder, one `aggregate.json` per experiment, flat PNG figures, and a root `run.json` with progress/status metadata under `eval_results/`.


In [1]:
import importlib
import os
import sys

# Run this notebook with the Jupyter/VS Code kernel named `venv`.
_EXPECTED_PREFIX = "/home/abd/Documents/project/venv"
if os.path.abspath(sys.prefix) != _EXPECTED_PREFIX: raise RuntimeError(f"Select the Jupyter kernel named `venv` before running HRC.ipynb. Current executable: {sys.executable}")

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-hrc")
print(f"Kernel executable: {sys.executable}")

from src.environment import StateTracker, gen
from src import experiments

print(f"{StateTracker().n_features} raw state features")
print(f"Default seeds: {tuple(experiments.DEFAULT_SEEDS)}")
print(f"Recipe catalog entries: {len(gen.recipe_library())}")
print("Experiment module: src.experiments")

for recipe_name in gen.recipe_library(): print(recipe_name)


Kernel executable: /home/abd/Documents/project/venv/bin/python
470 raw state features
Default seeds: (1337, 2024, 7, 9001, 31415)
Recipe catalog entries: 30
Experiment module: src.experiments
tomato_onion_soup_v1
tomato_soup
mushroom_soup
seasoned_mixture_soup
grilled_steak
burger
seasoned_chicken
garlic_fish
simple_salad
grated_cheese_salad
smoothie
yoghurt_smoothie
boiled_eggs
boiled_rice
tomato_garlic_soup
mushroom_garlic_soup
tomato_mushroom_soup
onion_rice_pot
tomato_cheese_salad
tomato_lettuce_salad
banana_strawberry_fruit_bowl
oil_tomato_salad
scrambled_eggs_pan
mushroom_omelette
meat_mushroom_skillet
garlic_chicken_salad
chicken_rice_plate
fish_rice_plate
yoghurt_fruit_bowl
rice_mushroom_bowl


## Review artifacts or explicitly run the suite


In [2]:
import json
from pathlib import Path

import src.models as _hrc_models
import src.memory as _hrc_memory
import src.adaptive_agent as _hrc_adaptive_agent
import src.baselines as _hrc_baselines

_hrc_models         = importlib.reload(_hrc_models)
_hrc_memory         = importlib.reload(_hrc_memory)
_hrc_adaptive_agent = importlib.reload(_hrc_adaptive_agent)
_hrc_baselines      = importlib.reload(_hrc_baselines)
experiments         = importlib.reload(experiments)

RUN_PUBLICATION_SUITE = True
INCLUDE_APPENDIX = True

run_config = experiments.RunConfig(
    seeds=experiments.DEFAULT_SEEDS,
    output_root="eval_results",
    timestamp_subdir=True,
    workers=len(experiments.DEFAULT_SEEDS),
    baselines=experiments.PAPER_BASELINES,
    profile=False,
    log_full_distributions=False,
    valid_action_expansion=False,
)

suite_config = experiments.ExperimentSuiteConfig(
    deployment_stream=experiments.DeploymentStreamConfig(n_recipes=10, n_users=4, n_phase_b_events=120),
    cross_recipe_transfer=experiments.TransferSuiteConfig(n_recipes=7, n_preferences=7, diagonal_cycles=1, offdiag_repeats=2),
    decay_reentry=experiments.DecayReentrySuiteConfig(n_recipes=12, gap_sweep=(5, 10, 15, 30, 45)),
    disambiguation_audit=experiments.DisambiguationAuditConfig(n_recipes=12),
    materiality_preflight=experiments.MaterialityPreflightConfig(n_recipes=15, n_preferences=7, min_effective_preferences=3),
)

def latest_completed_run(output_root):
    root = Path(output_root)
    for run_json in sorted(root.glob("*/run.json"), reverse=True):
        try:
            manifest = json.loads(run_json.read_text(encoding="utf-8"))
        except Exception:
            continue
        if (manifest.get("status") or {}).get("state") == "completed":
            return run_json.parent, manifest
    raise RuntimeError("No completed eval_results run found. Run the publication suite from the CLI after materiality_preflight passes.")

if RUN_PUBLICATION_SUITE:
    experiments_to_run = experiments.MAIN_EXPERIMENTS + (experiments.APPENDIX_EXPERIMENTS if INCLUDE_APPENDIX else ())
    print(f"Planned experiments: {len(experiments_to_run)} -> {experiments_to_run}")
    print(f"Seeds: {run_config.seeds}")
    print(f"Workers: {run_config.workers or len(run_config.seeds)}")
    print("Progress and ETA will stream below as each seed finishes.")
    result = experiments.run_suite(run_config=run_config, suite_config=suite_config, experiments=experiments_to_run)
else:
    artifact_root, run_manifest = latest_completed_run(run_config.output_root)
    status = run_manifest.get("status") or {}
    result = {
        "out_dir": str(artifact_root),
        "run_dir": str(artifact_root),
        "completed_experiments": list(status.get("completed", [])),
        "partial_experiments": list(status.get("partial", [])),
        "failed_experiments": {},
        "run_summary": ((run_manifest.get("result") or {}).get("run_summary") or {}),
    }
    print("Visualization mode: using latest completed run.")
    print(f"Artifact root: {artifact_root}")
    print("Set RUN_PUBLICATION_SUITE = True to regenerate the full publication suite.")


Planned experiments: 11 -> ('materiality_preflight', 'deployment_stream', 'cross_recipe_transfer', 'decay_reentry', 'disambiguation_audit', 'demo_count_sample_efficiency', 'deployment_ladder_heterogeneous_prefs', 'deployment_ladder_shared_pref', 'zipf_usage_sweep', 'adversarial_stress_suite', 'baseline_hyperparameter_tuning')
Seeds: (1337, 2024, 7, 9001, 31415)
Workers: 5
Progress and ETA will stream below as each seed finishes.
[materiality_preflight] start: 5 seeds, workers=5, historical batch estimate=0s; elapsed=0s ETA=15h45m (history=6/10 future, 4 fallback)
  [materiality_preflight] seed 1337 done in 0s (1/55, elapsed=1s ETA=15h44m (history=6/10 future, 4 fallback))
  [materiality_preflight] seed 2024 done in 0s (2/55, elapsed=1s ETA=15h44m (history=6/10 future, 4 fallback))
  [materiality_preflight] seed 9001 done in 1s (3/55, elapsed=1s ETA=15h44m (history=6/10 future, 4 fallback))
  [materiality_preflight] seed 31415 done in 1s (4/55, elapsed=1s ETA=15h44m (history=6/10 future

In [3]:
artifact_root = result.get("out_dir") or result.get("run_dir")
if artifact_root is None:
    raise RuntimeError("No output directory found in `result`. Run the experiment cell first.")
print("Completed experiments:")
for experiment_id in result.get("completed_experiments", []):
    print(f"  {experiment_id}")
if result.get("partial_experiments"):
    print("Partial experiments:")
    for experiment_id in result.get("partial_experiments", []):
        print(f"  {experiment_id}")
if result.get("failed_experiments"):
    print("Failed experiments:")
    for key, err in result["failed_experiments"].items():
        print(f"  {key}: {err}")
print()
print(f"Run JSON: {os.path.join(artifact_root, 'run.json')}")


Completed experiments:
  materiality_preflight
  deployment_stream
  decay_reentry
  disambiguation_audit
  demo_count_sample_efficiency
  deployment_ladder_heterogeneous_prefs
  deployment_ladder_shared_pref
  zipf_usage_sweep
  adversarial_stress_suite
  baseline_hyperparameter_tuning
Failed experiments:
  cross_recipe_transfer/seed_1337: RuntimeError: Cross-recipe transfer requires 7 recipes with 7 distinct effective preference orderings; run materiality_preflight and fix no-op preference transformations.
  cross_recipe_transfer/seed_7: RuntimeError: Cross-recipe transfer requires 7 recipes with 7 distinct effective preference orderings; run materiality_preflight and fix no-op preference transformations.
  cross_recipe_transfer/seed_31415: RuntimeError: Cross-recipe transfer requires 7 recipes with 7 distinct effective preference orderings; run materiality_preflight and fix no-op preference transformations.
  cross_recipe_transfer/seed_9001: RuntimeError: Cross-recipe transfer requi

In [4]:
from pathlib import Path

artifact_root = result.get("out_dir") or result.get("run_dir")
if artifact_root is None:
    raise RuntimeError("No output directory found in `result`. Run the experiment cell first.")
artifact_root = Path(artifact_root)
experiments.render_figures(artifact_root)
print("Artifacts written to:")
print(f"  {artifact_root}")
print()
for path in sorted(artifact_root.rglob("*")):
    if path.is_file() and path.suffix in {".json", ".png"}:
        print(path)


Artifacts written to:
  eval_results/2026-06-02_2337

eval_results/2026-06-02_2337/adversarial_stress_suite/aggregate/aggregate.json
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_1337/memory_exhaustion_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_1337/preference_thrash_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_1337/prefix_collision_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_1337/rare_reentry_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_2024/memory_exhaustion_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_2024/preference_thrash_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_2024/prefix_collision_stress_stress_score.png
eval_results/2026-06-02_2337/adversarial_stress_suite/seed_2024/rare_reentry_stress_stress_score.png
eval_results/2026-06-02_2337/ad